# 3. Main experiments

In [13]:
%load_ext autoreload
%autoreload 2

import os
import logging

import numpy as np
import pandas as pd
import torch.nn as nn

from ndbn.experiments import run_experiments
import ndbn.plotting as plotting
from ndbn.plotting import plot_grouped_curves
from config import *   # settings shared by every notebook (folders, DAG numbering, speed, ...)

logging.getLogger("pgmpy").setLevel(logging.ERROR)

# synthetic figures: the pseudocount-0.5 Dirichlet estimator is the matched-prior oracle (it is given the
# generating alpha), drawn in grey as a reference; Laplace is the realistic smoothed table
plotting.REPORT_LABELS["dirichlet"] = "Dirichlet (matched prior)"
plotting.REPORT_COLORS["dirichlet"] = "#8a8a8a"


In [14]:
# folders and RESULTS_CSV come from config.py; each cardinality's chosen NN configs are in TUNING_DIR,
# and the serial timing run has its own file
CHOSEN_JSON_C2 = os.path.join(TUNING_DIR, "chosen_nn_c2.json")
CHOSEN_JSON_C4 = os.path.join(TUNING_DIR, "chosen_nn_c4.json")
TIMING_CSV  = os.path.join(RESULTS_DIR, "results_timing.csv")
print(f"saving results to '{RESULTS_CSV}'")

saving results to 'results7-parallel\results.csv'


In [15]:
from IPython.display import HTML, display
display(HTML("""<style>
.cell-output-ipywidget-background { background-color: transparent !important; }
:root {
  --jp-widgets-color: var(--vscode-editor-foreground);
  --jp-widgets-font-size: var(--vscode-editor-font-size);
}
</style>"""))

In [16]:
# the final NN configs chosen in 2_hyperparameter_tuning.ipynb for each cardinality, best first:
# pick the ones to run by name with NN_NAMES in each section's settings cell
import json
print("Card=2 chosen configs:")
with open(CHOSEN_JSON_C2) as f:
    display(pd.DataFrame(json.load(f)["configs"]).set_index("name"))
print("\nCard=4 chosen configs:")
with open(CHOSEN_JSON_C4) as f:
    display(pd.DataFrame(json.load(f)["configs"]).set_index("name"))

Card=2 chosen configs:


,config_id,max_params_k10,lr,weight_decay,dropout,patience,batch_size,optimizer,n_epochs,hidden_dims,activation
name,,,,,,,,,,,
nn_256_128,76cb8405,742696,0.0003,0.01,0.1,20,128,adamw,1000,"[256, 128]",ReLU
nn_256,7589813f,89896,0.0006,0.01,0.1,50,128,adamw,1000,[256],ReLU
nn_128_64,5770cf4b,207528,0.0003,0.01,0.1,50,128,adamw,1000,"[128, 64]",ReLU
nn_128,ccbfffbc,44968,0.0006,0.01,0.1,50,128,adamw,1000,[128],ReLU
nn_64_32,34bc3fa8,62824,0.0006,0.01,0.1,50,128,adamw,1000,"[64, 32]",ReLU
nn_64,dbd55b7b,22504,0.0006,0.01,0.1,50,128,adamw,1000,[64],ReLU
nn_32,3e4d22b3,11272,0.0006,0.01,0.1,50,32,adamw,1000,[32],ReLU



Card=4 chosen configs:


,config_id,max_params_k10,lr,weight_decay,dropout,patience,batch_size,optimizer,n_epochs,hidden_dims,activation
name,,,,,,,,,,,
nn_256,f0c5e5f0,174416,0.0006,0.0,0.0,10,32,adamw,1000,[256],ReLU
nn_128_64,ae643bf1,247248,0.0006,0.0,0.3,50,32,adamw,1000,"[128, 64]",ReLU
nn_64_32,c6d024ed,82704,0.0006,0.0,0.3,50,32,adamw,1000,"[64, 32]",ReLU
nn_32_16,fc0b2513,31152,0.0010,0.0,0.3,50,32,adamw,1000,"[32, 16]",ReLU
nn_32,48990a9c,21872,0.0006,0.0,0.3,50,64,adamw,1000,[32],ReLU


## Main experiments (c=2)

In [8]:
import torch.nn as nn
from ndbn.experiments import load_chosen_configs

NODE_COUNTS_C2  = [10, 20, 30, 40, 50]
IN_DEGREES_C2   = [2, 4, 6, 8, 10]
ALPHAS_C2       = [0.5]
SIM_C2          = 5

CARD_C2          = 2
SAMPLE_SIZES_C2  = [200, 600, 1000, 1400]
VAL_FRAC_C2      = 0.3
N_TEST_C2        = 10000
N_EPOCHS_C2      = 200

BASELINES_C2 = [
    {"name": "mle",       "type": "mle"},
    {"name": "dirichlet", "type": "dirichlet", "pseudo_count": 0.5},   # matches the generator's alpha
    {"name": "laplace",   "type": "dirichlet", "pseudo_count": 1.0},   # Laplace (add-one): does not know alpha
    {"name": "nn_linear", "type": "nn", "hidden_dims": (), "activation": None},
]
NN_NAMES_C2 = None
CHOSEN_NN_C2 = load_chosen_configs(CHOSEN_JSON_C2)
if NN_NAMES_C2 is not None:
    unknown = set(NN_NAMES_C2) - {c["name"] for c in CHOSEN_NN_C2}
    if unknown:
        raise KeyError(f"not in {CHOSEN_JSON_C2}: {sorted(unknown)}; "
                       f"names there: {[c['name'] for c in CHOSEN_NN_C2]}")
    CHOSEN_NN_C2 = [c for c in CHOSEN_NN_C2 if c["name"] in NN_NAMES_C2]
ESTIMATORS_C2 = BASELINES_C2 + [{"type": "nn", **c} for c in CHOSEN_NN_C2]


In [10]:
run_experiments(
    node_counts=NODE_COUNTS_C2, in_degrees=IN_DEGREES_C2, alphas=ALPHAS_C2, n_dags=SIM_C2,
    sample_sizes=SAMPLE_SIZES_C2, estimators=ESTIMATORS_C2, card=CARD_C2,
    val_frac=VAL_FRAC_C2, n_test=N_TEST_C2, n_epochs=N_EPOCHS_C2,
    results_dir=RESULTS_DIR, verbose=VERBOSE, results_name=RESULTS_NAME,
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

DAGs:   0%|          | 0/125 [00:00<?, ?it/s]

## Main experiments (c=4)

In [18]:
import torch.nn as nn
from ndbn.experiments import load_chosen_configs

NODE_COUNTS  = [10,20,30] 
IN_DEGREES   = [2,4,6,8,10] # max_in_degree
ALPHAS       = [0.5] # sparse/peaked -> flat/uniform
SIM          = 5 # random DAGs per config

CARD         = 4 # results rows, data caches, resume checks and plots are all keyed on
                 # card, so runs with different cardinalities can share one results CSV
SAMPLE_SIZES     = [200,600, 1000, 1400] #train+val
VAL_FRAC         = 0.3
N_TEST           = 10000
N_EPOCHS         = 200


# baselines, always run
BASELINES = [
    {"name": "mle",       "type": "mle"},
    {"name": "dirichlet", "type": "dirichlet", "pseudo_count": 0.5},   # matches the generator's alpha
    # no Laplace at r=4: its k>=8 tables (~1M configurations per node) exhaust memory; run at r=2 only
    {"name": "nn_linear", "type": "nn", "hidden_dims": (), "activation": None},
]
# final NN config(s) chosen at the end of 2_hyperparameter_tuning.ipynb (saved to CHOSEN_JSON_C4,
# shown at the top of this notebook). NN_NAMES = None runs all of them; a list of names
# (e.g. ["nn_256"]) runs only those
NN_NAMES = None #["nn_32"]
CHOSEN_NN = load_chosen_configs(CHOSEN_JSON_C4)
if NN_NAMES is not None:
    unknown = set(NN_NAMES) - {c["name"] for c in CHOSEN_NN}
    if unknown:
        raise KeyError(f"not in {CHOSEN_JSON_C4}: {sorted(unknown)}; "
                       f"names there: {[c['name'] for c in CHOSEN_NN]}")
    CHOSEN_NN = [c for c in CHOSEN_NN if c["name"] in NN_NAMES]
ESTIMATORS = BASELINES + [{"type": "nn", **c} for c in CHOSEN_NN]


In [19]:
run_experiments(
    node_counts=NODE_COUNTS, in_degrees=IN_DEGREES, alphas=ALPHAS, n_dags=SIM,
    sample_sizes=SAMPLE_SIZES, estimators=ESTIMATORS, card=CARD,
    val_frac=VAL_FRAC, n_test=N_TEST, n_epochs=N_EPOCHS,
    results_dir=RESULTS_DIR, verbose=VERBOSE, results_name=RESULTS_NAME,
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

DAGs:   0%|          | 0/75 [00:00<?, ?it/s]

## Structured ground truth (c=2)
Logistic CPDs instead of Dirichlet draws.

In [ ]:
from ndbn.generation import logistic_scale_matching

STRUCT_CARD          = 2
STRUCT_NODE_COUNTS   = [20]
STRUCT_IN_DEGREES    = [2, 4, 6, 8, 10]
STRUCT_SAMPLE_SIZES  = [200, 600, 1000, 1400]   # train + val, as the main grid
STRUCT_INTERACTIONS  = [0.0, 0.5]               # share of score variance from pairwise parent terms
STRUCT_SCALE         = round(logistic_scale_matching(alpha=0.5, card=STRUCT_CARD), 4)   # matched to Dirichlet(0.5) sharpness
STRUCT_RESULTS       = "results_structured.csv"
STRUCT_CSV           = os.path.join(RESULTS_DIR, STRUCT_RESULTS)

# fixed tabular references, the validation-tuned Dirichlet, and the r=2 networks of the main grid
STRUCT_ESTIMATORS = (
    [{"name": "mle",             "type": "mle"},
     {"name": "dirichlet",       "type": "dirichlet", "pseudo_count": 0.5},
     {"name": "laplace",         "type": "dirichlet", "pseudo_count": 1.0},
     {"name": "dirichlet_tuned", "type": "dirichlet_tuned"}]          # pseudocount from mle.PSEUDO_COUNT_GRID
    + [e for e in ESTIMATORS_C2 if e["name"] in ("nn_linear", "nn_32")]
)
print(f"logit scale {STRUCT_SCALE}; estimators {[e['name'] for e in STRUCT_ESTIMATORS]}")

In [ ]:
# resumable: (estimator, DAG, k, N, interaction) already in STRUCT_CSV are skipped
for interaction in STRUCT_INTERACTIONS:
    run_experiments(
        node_counts=STRUCT_NODE_COUNTS, in_degrees=STRUCT_IN_DEGREES, alphas=[0.5], n_dags=SIM_C2,
        sample_sizes=STRUCT_SAMPLE_SIZES, estimators=STRUCT_ESTIMATORS, card=STRUCT_CARD,
        val_frac=VAL_FRAC_C2, n_test=N_TEST_C2, n_epochs=N_EPOCHS_C2,
        results_dir=RESULTS_DIR, verbose=VERBOSE, results_name=STRUCT_RESULTS,
        n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
        generator={"kind": "logistic", "scale": STRUCT_SCALE, "interaction": interaction},
    )

In [ ]:
# report figure: KL vs in-degree per interaction share, and the summary the text quotes
if os.path.exists(STRUCT_CSV):
    st = pd.read_csv(STRUCT_CSV)
    # reference panel: the independent-column ground truths on the same 20-node DAGs and sample sizes
    main = pd.read_csv(RESULTS_CSV)
    indep = main[(main["card"].fillna(2) == STRUCT_CARD) & (main.n_nodes.isin(STRUCT_NODE_COUNTS))
                 & (main.alpha == 0.5) & (main.sample_size.isin(STRUCT_SAMPLE_SIZES))]
    plotting.plot_structured(st, [e["name"] for e in STRUCT_ESTIMATORS], independent=indep,
                             out_path=os.path.join(FIG_DIR, "structured.png"))
    p = st.pivot_table(index=["interaction", "in_degree", "sample_size", "dag_idx"], columns="estimator", values="kl")
    for base in ["dirichlet_tuned", "dirichlet", "laplace", "nn_linear"]:
        print(f"nn_32 beats {base}: " + ", ".join(
            f"interaction {lam:g}: {(g['nn_32'] < g[base]).sum()}/{len(g)}" for lam, g in p.groupby(level="interaction")))
    display(p.groupby(level=["interaction", "in_degree"]).mean().round(3))
    display(st[st.estimator == "dirichlet_tuned"].groupby(["interaction", "in_degree"]).pseudo_count.agg(lambda s: s.value_counts().to_dict()))
else:
    print(f"no {STRUCT_RESULTS} yet: run the cell above")

## Plots

In [ ]:
# Values held constant in each plot (everything except x and hue)
plotting.DEFAULT_SAMPLE_SIZE = 1000
plotting.DEFAULT_ALPHA       = 0.5
plotting.DEFAULT_N_NODES     = 20
plotting.DEFAULT_IN_DEGREE   = 4
plotting.DEFAULT_ESTIMATOR   = "mle"
# estimators plotted: the chosen configs in CHOSEN_JSON_C4 plus the mle / nn_linear baselines
plotting.DEFAULT_ESTIMATORS  = CHOSEN_JSON_C4


### Report figures

In [ ]:
# report figure (Methodology): a CPT next to the neural CPD that replaces it, for one node with two parents
plotting.plot_neural_cpd_diagram(out_path=os.path.join(FIG_DIR, "neural_cpd.png"))

In [ ]:
# report figures for every cardinality in FIG_CARDS, saved to FIG_DIR as <name>_c<card>.png (rows from
# both cardinalities share the CSV, so each figure uses only its own card's rows)
from ndbn.experiments import load_chosen_configs

FIG_CARDS = [2, 4]
MAIN_NN   = "nn_32"                                          # the NN compared with the tabular estimators
TABULAR   = ["mle", "dirichlet"]
# Laplace (run at r=2 only) is the realistic smoothed table; the Dirichlet estimator is the matched-prior oracle
MAIN_ESTS = ["mle", "laplace", "dirichlet", "nn_linear", MAIN_NN]
FIG_NODES = 30   # KL is summed over nodes, so figures that pool settings use one network size

df = pd.read_csv(RESULTS_CSV)
for card in FIG_CARDS:
    rep = df[df["card"].fillna(2) == card]
    fig_path = lambda name: os.path.join(FIG_DIR, f"{name}_c{card}.png")
    # the tuned NNs chosen for this cardinality
    chosen = load_chosen_configs(os.path.join(TUNING_DIR, f"chosen_nn_c{card}.json"))
    present = set(rep.estimator)
    ALL_ESTS = [e for e in ["mle", "laplace", "dirichlet", "nn_linear"] + [c["name"] for c in chosen] if e in present]
    MAIN_CARD = [e for e in MAIN_ESTS if e in present]

    plotting.plot_mle_fails(rep, estimators=TABULAR, sample_size=1000, alpha=0.5, logy=False,
                           out_path=fig_path("mle_fails"))
    plotting.plot_estimators_by_k(rep, ALL_ESTS, param_nodes=20, kl_nodes=FIG_NODES, out_path=fig_path("arms"))
    plotting.plot_kl_vs_n(rep[rep.n_nodes == FIG_NODES], MAIN_CARD, out_path=fig_path("kl_vs_N"))
    plotting.plot_size_alpha(rep, MAIN_CARD, out_path=fig_path("size_alpha"))

# rows per parent configuration: both cardinalities on one figure, one line per in-degree, pooled over
# the network sizes both cardinalities were run at
# the report draws MLE only, as plain points, at r=2 (every network size); the table also gives the smoothed
# tables' ratios, which the text quotes
_, rho_table = plotting.plot_ratio_vs_rho(df[df["card"].fillna(2) == 2],
                                          baselines=["mle", "dirichlet", "laplace"], panels="mle", by_k=False, logy=False,
                                          estimator=MAIN_NN, alpha=0.5,
                                          out_path=os.path.join(FIG_DIR, "rho.png"))
rho_table.round(2)

In [ ]:
# initial check on the ground-truth distribution: an earlier run over Dirichlet alpha 0.1, 0.5 and 1.0 (untuned nn_32,
# 5-40 nodes, in-degrees 2-10, N in 600-1400, five DAGs each), kept in alphas.csv; mean KL by network size and by alpha
alphas = pd.read_csv(os.path.join(os.path.dirname(RESULTS_CSV), "alphas.csv"))
plotting.plot_size_alpha(alphas, ["mle", "nn_linear", "nn_32"], out_path=os.path.join(FIG_DIR, "alpha_check.png"))
alphas.groupby(["alpha", "estimator"]).kl.mean().unstack().round(2)

In [ ]:
# Table 4 (rows needed to reach KL <= 5 nats, r=2, FIG_NODES-node networks) and Supplementary Figure S5
# (KL vs N for every network size: pooling over sizes only rescales KL, the ordering stays the same)
rep2 = df[df["card"].fillna(2) == 2]
display(plotting.rows_to_reach(rep2[rep2.n_nodes == FIG_NODES], ["mle", "nn_linear", MAIN_NN, "dirichlet"], threshold=5.0))
plotting.plot_kl_vs_n_by_size(rep2, MAIN_ESTS, sizes=[10, 20, 30, 50], out_path=os.path.join(FIG_DIR, "kl_vs_N_by_size.png"))

In [ ]:
# Supplementary Table S3: mean KL divergence +- standard error over the five DAGs, r=2, FIG_NODES-node networks,
# for each in-degree and sample size (the values behind Figures 3a and 4); written for \input in supplementary.tex
TABLE_DIR = os.path.join("report", "tables")
rep2 = df[(df["card"].fillna(2) == 2) & (df.n_nodes == FIG_NODES) & (df.alpha == 0.5)]
tex = plotting.mean_err_table(rep2, ["in_degree", "sample_size"], ["mle", "laplace", "dirichlet", "nn_linear", MAIN_NN],
                              value="kl", err="se", index_labels=["$k$", "$N$"],
                              out_path=os.path.join(TABLE_DIR, "synthetic_kl_r2.tex"))
print(tex)


In [ ]:
# how much MLE's KL, and its ranking against nn_32, depends on the floor on its zero probabilities
# (report Section 3.7): MLE refitted on the cached data with each floor in experiments.MLE_FLOORS.
# nn_32 needs no floor (a softmax never outputs zero), so its KL comes from the main results.
from ndbn.experiments import run_mle_floor_check

RUN_MLE_FLOOR = False   # True to (re)run (resumable; about 30 min serially at r=2); False uses the saved CSV
MLE_FLOOR_CSV = os.path.join(RESULTS_DIR, "results_mle_floor.csv")
if RUN_MLE_FLOOR:
    run_mle_floor_check(NODE_COUNTS_C2, IN_DEGREES_C2, ALPHAS_C2, SIM_C2, SAMPLE_SIZES_C2, CARD_C2,
                        N_TEST_C2, RESULTS_DIR, n_jobs=1)

fl = pd.read_csv(MLE_FLOOR_CSV)
key = ["n_nodes", "in_degree", "alpha", "dag_idx", "sample_size"]
nn = df[(df.estimator == MAIN_NN) & (df["card"].fillna(2) == 2)][key + ["kl"]].rename(columns={"kl": "kl_nn"})
m = fl[fl.card == 2].merge(nn, on=key)
m["nn_wins"] = m.kl_nn < m.kl
print("share of configurations where nn_32 beats MLE, by floor and in-degree")
display(m.pivot_table(index="floor", columns="in_degree", values="nn_wins", aggfunc="mean").round(3))
display(m.assign(ratio=m.kl / m.kl_nn).groupby("floor")
         .agg(wins=("nn_wins", "sum"), configs=("nn_wins", "size"), median_ratio=("ratio", "median")).round(2))
# the rho figure at each floor (r=2, every network size): the floor scales the ratios but keeps their shape
m["rho"] = m.sample_size / 2 ** m.in_degree
display(m.assign(ratio=m.kl / m.kl_nn).groupby(["floor", "in_degree", "sample_size"])
         .agg(rho=("rho", "first"), ratio=("ratio", "mean")).reset_index()
         .pivot_table(index=["in_degree", "sample_size", "rho"], columns="floor", values="ratio").round(2))


In [ ]:
# §4.1: share of parent configurations never seen in training (20-node networks, r=2, N=1000; mean over DAGs)
from ndbn.experiments import load_dag
from ndbn.evaluation import unseen_config_share

unseen = pd.DataFrame([
    {"in_degree": k, "dag": d,
     "unseen_share": unseen_config_share(G, train.iloc[:1000], 2)}
    for k in [2, 4, 6, 8, 10] for d in range(1, SIM + 1)
    for G, _, train, _ in [load_dag(RESULTS_DIR, d, k, 20, 0.5, 2, 1400)]
])
unseen.groupby("in_degree").unseen_share.mean().mul(100).round(1).rename("never seen (%)")

### Timing run (serial)
Run on an idle machine.

In [ ]:
TIMING_DESIGN = dict(node_counts=[20], in_degrees=[2, 6, 10], n_dags=1, alphas=[0.5],
                     sample_sizes=[600, 1400], n_test=N_TEST, results_dir=RESULTS_DIR, verbose=VERBOSE,
                     results_name="results_timing.csv", n_jobs=1, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS)
TIMED = {"mle", "dirichlet", "laplace", "nn_linear", "nn_32"}
for card, ests, val_frac, n_epochs in [(CARD_C2, ESTIMATORS_C2, VAL_FRAC_C2, N_EPOCHS_C2),
                                      (CARD, ESTIMATORS, VAL_FRAC, N_EPOCHS)]:
    run_experiments(estimators=[e for e in ests if e["name"] in TIMED], card=card,
                    val_frac=val_frac, n_epochs=n_epochs, **TIMING_DESIGN)

DAGs:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\Kaylin\Documents\Neural-Discrete-Bayesian-Network\ndbn\models.py:66: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  idx = torch.as_tensor(data[parents].to_numpy(np.int64), device=DEVICE)


KeyboardInterrupt: 